In [35]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

In [36]:
# Create a DataFrame from Table 9 data
data = {
    "Model": [
        "AttGRU (Original)",
        "GRU without Attention",
        "LSTM with Attention",
        "Bidirectional AttGRU",
        "Simple MLP",
        "Deep AttGRU (2 layers)"
    ],
    "ACC": [0.9790, 0.9054, 0.9615, 0.9839, 0.9100, 0.8795],
    "AUC": [0.9857, 0.9850, 0.9858, 0.9845, 0.9883, 0.9838],
    "PRE": [0.9066, 0.9043, 0.9080, 0.9008, 0.9094, 0.8743],
    "SP": [0.9029, 0.8978, 0.9046, 0.8962, 0.9024, 0.8699],
    "SN": [0.9040, 0.9686, 0.9706, 0.9682, 0.9702, 0.9596],
    "F1": [0.8795, 0.8997, 0.9057, 0.8977, 0.9039, 0.8696],
    "MCC": [0.9790, 0.8737, 0.8815, 0.8713, 0.8801, 0.8394]
}

In [37]:
df = pd.DataFrame(data)

In [38]:
# Let's add short names for the models for better visualization
short_names = {
    "AttGRU (Original)": "AttGRU",
    "GRU without Attention": "GRU - No Attn",
    "LSTM with Attention": "LSTM + Attn",
    "Bidirectional AttGRU": "Bi-AttGRU",
    "Simple MLP": "MLP",
    "Deep AttGRU (2 layers)": "Deep AttGRU"
}

df['Short_Name'] = df['Model'].map(short_names)

In [39]:
# Reset matplotlib settings to default
plt.rcParams.update(plt.rcParamsDefault)

# Set Times New Roman font with fallback
plt.rcParams['font.family'] = ['Times New Roman', 'serif']
plt.rcParams['mathtext.fontset'] = 'stix'

# Set global plot style with large font sizes
plt.rcParams['font.size'] = 32
plt.rcParams['axes.labelsize'] = 36
plt.rcParams['axes.titlesize'] = 40
plt.rcParams['xtick.labelsize'] = 30
plt.rcParams['ytick.labelsize'] = 30
plt.rcParams['legend.fontsize'] = 22  # Reduced legend font size
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['savefig.dpi'] = 1000
plt.rcParams['figure.facecolor'] = 'white'

In [40]:
# Use the feature_selection_palette for model variants
model_palette = {
    "AttGRU": "#FF5733",
    "GRU - No Attn": "#33FF57",
    "LSTM + Attn": "#3357FF",
    "Bi-AttGRU": "#FF33A1",
    "MLP": "#FFA133",
    "Deep AttGRU": "#33FFA1"
}

In [41]:
# Function to save figures in both PNG and PDF in a dedicated folder
def save_figure(fig, filename):
    output_folder = "Experiment_Output_Figures/Ablation_Study"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    png_path = os.path.join(output_folder, f"{filename}.png")
    pdf_path = os.path.join(output_folder, f"{filename}.pdf")
    
    fig.savefig(png_path, dpi=1000, bbox_inches='tight', facecolor='white', format='png')
    fig.savefig(pdf_path, dpi=1000, bbox_inches='tight', facecolor='white', format='pdf')
    
    print(f"Saved: {png_path} and {pdf_path}")
    plt.close(fig)

In [42]:
# Function to generate the violin plot visualization
def generate_violin_plot():
    metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
    
    melted_df = pd.melt(df, id_vars=['Short_Name'], 
                        value_vars=metrics, 
                        var_name='Metric', 
                        value_name='Score')
    
    fig, ax = plt.subplots(figsize=(20, 14))
    
    sns.violinplot(x='Short_Name', 
                   y='Score', 
                   hue='Short_Name',  # Assign hue to Short_Name
                   data=melted_df, 
                   palette=[model_palette[model] for model in df['Short_Name']], 
                   inner='box', 
                   linewidth=2, 
                   ax=ax, 
                   legend=False)  # Disable legend since hue matches x
    
    ax.set_title('Distribution of Performance Scores by Model Variant', 
                fontsize=44, pad=30)
    ax.set_xlabel('Model Variant', fontsize=40, labelpad=25)
    ax.set_ylabel('Score Distribution Across Metrics', fontsize=40, labelpad=25)
    
    ax.set_ylim(0.75, 1.1)  # Adjusted for the data range
    
    # ax.grid(axis='y', linestyle='--', alpha=0.7)  # Commented out as in your code
    
    plt.tight_layout(pad=3.0)
    
    save_figure(fig, "ablation_violin_plot")

In [43]:
# Function to generate the timing comparison plot
def generate_timing_comparison():
    fig, ax = plt.subplots(figsize=(20, 14))
    
    sorted_df = df.sort_values(by='Training_Time', ascending=True)
    
    bars = ax.barh(sorted_df['Short_Name'], sorted_df['Training_Time'], 
                  color=[model_palette[model] for model in sorted_df['Short_Name']], 
                  height=0.6)
    
    for i, model in enumerate(sorted_df['Short_Name']):
        test_time = sorted_df[sorted_df['Short_Name'] == model]['Testing_Time'].values[0]
        ax.text(3, i, f'Test: {test_time:.4f}s', ha='left', va='center', 
                fontsize=22, color='white')
    
    ax.set_title('Training and Testing Time by Model Variant', fontsize=40, pad=30)
    ax.set_xlabel('Training Time (seconds)', fontsize=36, labelpad=25)
    ax.set_ylabel('Model Variant', fontsize=36, labelpad=25)
    
    max_time = sorted_df['Training_Time'].max()
    ax.set_xlim(0, max_time * 1.2)
    
    # ax.grid(axis='x', linestyle='--', alpha=0.7)
    
    plt.tight_layout(pad=3.0)
    
    save_figure(fig, "ablation_timing_comparison")

In [44]:
# Function to generate a performance comparison chart
def generate_performance_comparison():
    metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
    
    melted_df = pd.melt(df, id_vars=['Short_Name'], 
                    value_vars=metrics, 
                    var_name='Metric', 
                    value_name='Score')
    
    fig, ax = plt.subplots(figsize=(20, 14))
    
    sns.barplot(x='Metric', y='Score', hue='Short_Name', 
                data=melted_df, 
                palette=[model_palette[model] for model in df['Short_Name']], 
                ax=ax, edgecolor='none')
    
    ax.set_title('Comparison of Model Variants across Metrics', fontsize=44, pad=30)
    ax.set_xlabel('Evaluation Metric', fontsize=40, labelpad=25)
    ax.set_ylabel('Score', fontsize=40, labelpad=25)
    ax.set_ylim(0, 1)  # Adjusted for the data range
    
    ax.legend(title='Model Variant', title_fontsize=24, fontsize=22, 
              bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout(pad=3.0)
    
    save_figure(fig, "ablation_performance_comparison_grouped_bar")

In [45]:
# Function to generate a radar chart
def generate_radar_chart():
    metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
    
    categories = metrics
    N = len(categories)
    
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))
    
    for i, idx in enumerate(df.index):
        model = df.loc[idx]
        values = model[metrics].values.tolist()
        values += values[:1]
        
        ax.plot(angles, values, linewidth=4, label=model['Short_Name'], 
                color=model_palette[model['Short_Name']])
        ax.fill(angles, values, alpha=0.1, color=model_palette[model['Short_Name']])
    
    ax.set_ylim(0, 1)  # Adjusted for the data range
    
    plt.xticks(angles[:-1], categories, fontsize=36)
    
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=22)
    
    plt.title('Model Variants Comparison (Radar Chart)', 
              fontsize=44, pad=40, y=1.08)
    
    plt.tight_layout(pad=3.0)
    
    save_figure(fig, "ablation_radar_chart")

In [46]:
# Function to generate a heatmap
def generate_heatmap():
    metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
    
    heatmap_df = df[['Short_Name'] + metrics].set_index('Short_Name')
    
    fig, ax = plt.subplots(figsize=(20, 14))
    
    heatmap = sns.heatmap(heatmap_df, annot=True, fmt=".4f", cmap="YlGnBu", 
                          linewidths=0.5, linecolor='white', annot_kws={'size': 26},
                          vmin=0.9, vmax=1.0)  # Adjusted for the data range
    
    cbar = heatmap.collections[0].colorbar
    cbar.set_label('Score', size=34)
    cbar.ax.tick_params(labelsize=28)
    
    ax.set_title('Model Variants Performance Heatmap', 
                 fontsize=44, pad=30)
    ax.set_xlabel('Evaluation Metrics', fontsize=40, labelpad=25)
    ax.set_ylabel('Model Variant', fontsize=40, labelpad=25)
    
    plt.tight_layout(pad=3.0)
    
    save_figure(fig, "ablation_heatmap")

In [47]:
# Function to generate individual metric charts
def generate_individual_metric_charts():
    metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
    
    for metric in metrics:
        fig, ax = plt.subplots(figsize=(16, 12))
        
        sorted_df = df.sort_values(by=metric, ascending=False)
        
        bars = ax.bar(sorted_df['Short_Name'], sorted_df[metric], 
                     color=[model_palette[model] for model in sorted_df['Short_Name']], width=0.6)
        
        ax.set_title(f'{metric} Scores by Model Variant', 
                    fontsize=40, pad=30)
        ax.set_xlabel('Model Variant', fontsize=36, labelpad=25)
        ax.set_ylabel(f'{metric} Score', fontsize=36, labelpad=25)
        
        plt.xticks(rotation=45, ha='right')
        
        ax.set_ylim(0, 1)  # Adjusted for the data range
        
        # ax.grid(axis='y', linestyle='--', alpha=0.7)
        
        plt.tight_layout(pad=3.0)
        
        save_figure(fig, f"ablation_{metric}_comparison")

In [48]:
generate_violin_plot()

Saved: Experiment_Output_Figures/Ablation_Study/ablation_violin_plot.png and Experiment_Output_Figures/Ablation_Study/ablation_violin_plot.pdf


In [49]:
# generate_timing_comparison()

In [50]:
generate_performance_comparison()

Saved: Experiment_Output_Figures/Ablation_Study/ablation_performance_comparison_grouped_bar.png and Experiment_Output_Figures/Ablation_Study/ablation_performance_comparison_grouped_bar.pdf


In [51]:
generate_radar_chart()

Saved: Experiment_Output_Figures/Ablation_Study/ablation_radar_chart.png and Experiment_Output_Figures/Ablation_Study/ablation_radar_chart.pdf


In [52]:
generate_heatmap()

Saved: Experiment_Output_Figures/Ablation_Study/ablation_heatmap.png and Experiment_Output_Figures/Ablation_Study/ablation_heatmap.pdf


In [53]:
generate_individual_metric_charts()

Saved: Experiment_Output_Figures/Ablation_Study/ablation_ACC_comparison.png and Experiment_Output_Figures/Ablation_Study/ablation_ACC_comparison.pdf
Saved: Experiment_Output_Figures/Ablation_Study/ablation_AUC_comparison.png and Experiment_Output_Figures/Ablation_Study/ablation_AUC_comparison.pdf
Saved: Experiment_Output_Figures/Ablation_Study/ablation_PRE_comparison.png and Experiment_Output_Figures/Ablation_Study/ablation_PRE_comparison.pdf
Saved: Experiment_Output_Figures/Ablation_Study/ablation_SP_comparison.png and Experiment_Output_Figures/Ablation_Study/ablation_SP_comparison.pdf
Saved: Experiment_Output_Figures/Ablation_Study/ablation_SN_comparison.png and Experiment_Output_Figures/Ablation_Study/ablation_SN_comparison.pdf
Saved: Experiment_Output_Figures/Ablation_Study/ablation_F1_comparison.png and Experiment_Output_Figures/Ablation_Study/ablation_F1_comparison.pdf
Saved: Experiment_Output_Figures/Ablation_Study/ablation_MCC_comparison.png and Experiment_Output_Figures/Ablati

In [54]:
print("All ablation study visualizations have been generated!")
print(f"Files are saved in the 'experiment_output_figures/ablation_table9' folder")

All ablation study visualizations have been generated!
Files are saved in the 'experiment_output_figures/ablation_table9' folder
